In [1]:
# Initial planet parameters (Alderson et al. 2022)
# These baseline values will be used as the starting PSG parameters for the manual example.
Teq = 1770.0  # equilibrium temperature [K]
Rp_J = 1.99   # planet radius [R_jup]
Mp_J = 0.486  # planet mass [M_jup]
T_mid_reference = 1724.18  # mid-transit atmospheric temperature from Alderson et al. 2022
# Note: the pipeline interpolates PSG CFG tags such as 'ATMOSPHERE-TEMPERATURE'.
# The mid-transit reference temperature is the actual baseline used for cfg interpolation.


# PSG Asymmetry Pipeline
This notebook generates the Asym_* spectrum folders by interpolating between the reference mid-transit configuration and the extreme ingress/egress models, then downloads the spectra from NASA PSG.

In [2]:
import numpy as np
import os
import re
import requests

In [3]:
PSG_URL = "https://psg.gsfc.nasa.gov/api.php"
cfg_templates = {
    "ingress": "Asym_1.0/wasp17b_transit_ingress_ultra_hazy_cold.cfg",
    "mid": "Asym_1.0/wasp17b_transit_mid_ultra_reference.cfg",
    "egress": "Asym_1.0/wasp17b_transit_egress_ultra_clear_hot.cfg"}
PARAMS = ["ATMOSPHERE-TEMPERATURE", "ATMOSPHERE-CLOUDFRACTION", "ATMOSPHERE-AABUN",]

In [4]:
def run_psg(cfg_text):
    response = requests.post(
        PSG_URL,
        data=cfg_text.encode("utf-8"),
        headers={"User-Agent": "Mozilla/5.0", "Content-Type": "text/plain"}, timeout=300)
    text = response.text
    if response.status_code != 200:
        raise RuntimeError(f"PSG error: {response.status_code}")
    if "# Wave/freq" not in text and "#Wave/freq" not in text:
        raise RuntimeError("PSG returned response without spectrum")
    return text
def extract_spectrum(psg_html):
    start_patterns = ["# Wave/freq", "#Wave/freq"]
    lines = psg_html.splitlines()
    start_idx = -1
    for i, line in enumerate(lines):
        if any(p in line for p in start_patterns):
            start_idx = i
            break
    if start_idx == -1:
        raise RuntimeError("Spectrum header not found")
    data = []
    for l in lines[start_idx:]:
        l = l.strip()
        if not l:
            continue
        if l.startswith("<"):
            continue
        if l.startswith("#") and not l[0].isdigit():
            continue
        parts = l.split()
        if len(parts) < 5:
            continue
        try:
            float(parts[0])
        except ValueError:
            continue
        data.append(l)
    from io import StringIO
    return np.loadtxt(StringIO("\n".join(data)))

In [5]:
def get_value(cfg, tag):
    m = re.search(rf"<{tag}>([-+0-9.eE]+)", cfg)
    return float(m.group(1)) if m else None
def set_value(cfg, tag, val):
    return re.sub(rf"<{tag}>[-+0-9.eE]+", f"<{tag}>{val}", cfg)
def interpolate(mid_cfg, ext_cfg, a):
    out = mid_cfg
    for p in PARAMS:
        mid_v = get_value(mid_cfg, p)
        ext_v = get_value(ext_cfg, p)
        if mid_v is None or ext_v is None:
            continue
        new_v = mid_v + a * (ext_v - mid_v)
        out = set_value(out, p, new_v)
    return out

In [6]:
def run_literature_symmetric():
    with open(cfg_templates["mid"], "r", encoding="utf-8") as f:
        cfg = f.read()
    cfg = set_value(cfg, "ATMOSPHERE-TEMPERATURE", 1770.0)
    cfg = set_value(cfg, "ATMOSPHERE-CLOUDFRACTION", 0.50)
    cfg = set_value(cfg, "ATMOSPHERE-AABUN", 0.08)
    folder = "Literature_Alderson_Symmetric"
    os.makedirs(folder, exist_ok=True)
    for phase in ["ingress","mid","egress"]:
        with open(os.path.join(folder,f"{phase}.cfg"),"w",encoding="utf-8") as f:
            f.write(cfg)
        html = run_psg(cfg)
        data = extract_spectrum(html)
        np.savetxt(os.path.join(folder,f"{phase}.txt"),data)
    print("Literature_Alderson_Symmetric COMPLETE")
def run_literature_mild_asymmetry():
    with open(cfg_templates["mid"], "r", encoding="utf-8") as f:
        cfg_mid = f.read()
    cfg_ing = cfg_mid
    cfg_eg = cfg_mid
    cfg_ing = set_value(cfg_ing,"ATMOSPHERE-TEMPERATURE",1700.0)
    cfg_ing = set_value(cfg_ing,"ATMOSPHERE-CLOUDFRACTION",0.60)
    cfg_ing = set_value(cfg_ing,"ATMOSPHERE-AABUN",0.10)
    cfg_mid = set_value(cfg_mid,"ATMOSPHERE-TEMPERATURE",1770.0)
    cfg_mid = set_value(cfg_mid,"ATMOSPHERE-CLOUDFRACTION",0.50)
    cfg_mid = set_value(cfg_mid,"ATMOSPHERE-AABUN",0.08)
    cfg_eg = set_value(cfg_eg,"ATMOSPHERE-TEMPERATURE",1840.0)
    cfg_eg = set_value(cfg_eg,"ATMOSPHERE-CLOUDFRACTION",0.40)
    cfg_eg = set_value(cfg_eg,"ATMOSPHERE-AABUN",0.06)
    folder = "Literature_Alderson_Mild_Asymmetry"
    os.makedirs(folder, exist_ok=True)
    scenarios = {"ingress":cfg_ing,"mid":cfg_mid,"egress":cfg_eg}
    for phase,cfg in scenarios.items():
        with open(os.path.join(folder,f"{phase}.cfg"),"w",encoding="utf-8") as f:
            f.write(cfg)
        html = run_psg(cfg)
        data = extract_spectrum(html)
        np.savetxt(os.path.join(folder,f"{phase}.txt"),data)
    print("Literature_Alderson_Mild_Asymmetry COMPLETE")

In [7]:
#RUN
run_literature_symmetric()
run_literature_mild_asymmetry()

Literature_Alderson_Symmetric COMPLETE
Literature_Alderson_Mild_Asymmetry COMPLETE
